## 第 2 周第 3 天作业 —— Shopify + Rails 代码助手

### 练习目标

做一个聊天机器人：帮你在**现有 Ruby on Rails 代码库**里集成 **Shopify API**（生成模型、helper、脚手架命令等）。

### 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Gradio 聊天 UI | `gr.ChatInterface(..., type="messages")` |
| OpenAI 消息格式 | `system` + 历史 `history` + 当前 `user` |
| 本地 Ollama | 用 OpenAI 兼容客户端指向 `localhost:11434/v1`，模型 `llama3.2` |
| 动态 system prompt | 用户消息含 `shopify` 时追加脚手架指引 |

### 怎么跑

1. 先启动本机 Ollama，并确保已拉取 `llama3.2`
2. 从上到下依次运行单元格
3. 在 Gradio 界面里提问（例如如何创建 Shopify 相关 model）


In [ ]:
# ========== 导入：后面要用的库 ==========

# 标准库 os：读环境变量等（本格主要占位，后面也常用）
import os
# Gradio：快速搭聊天 UI（ChatInterface）
import gradio as gr
# OpenAI 官方 Python SDK：这里用来对接「OpenAI 兼容」的本地 Ollama
from openai import OpenAI


In [ ]:
# ========== 本地 Ollama 客户端（OpenAI 兼容接口） ==========

# Ollama 的 OpenAI 兼容基址：注意路径带 /v1
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 本地模型名：须与 ollama list 里已安装的名字一致
MODEL = "llama3.2"
# 创建客户端：base_url 指到本机；api_key 对 Ollama 通常任意非空字符串即可
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== system prompt：定助手人设与回答风格 ==========
# 发给模型的指令字符串保持英文原文（翻译会改变模型行为）

system_message = """
You are a helpful assistant that does Shopify API integrations in Ruby on Rails codebases.
Give short precise answers and include the code blocks that the user asks for.
If the user doesn't know where to start guide them to the models and helpers that need to be created.
Always be accurate. If you don't know the answer, say so.
"""


In [ ]:
# ========== 聊天回调 + 启动 Gradio ==========

def chat(message, history):
    # 默认用上面的 system_message；若用户提到 shopify 再追加脚手架指引
    relevant_system_message = system_message
    # 不区分大小写：消息里出现 shopify 就加一段「给 scaffolding 命令」的补充说明
    if 'shopify' in message.lower():
        # 追加的英文同样是发给模型的指令，保持原样不翻译
        relevant_system_message += """
            You provide the Ruby on Rails scaffolding commands to build each database
            model needed to integrate Shopify API to the existing Ruby on Rails codebase.
        """

    # Gradio type="messages" 时 history 是 [{role, content}, ...]；这里再规范成 OpenAI 消息列表
    chat_history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 拼完整 messages：system → 历史 → 当前 user
    messages = [{"role": "system", "content": relevant_system_message}] + chat_history + [{"role": "user", "content": message}]
    # 调本地 Ollama（经 OpenAI 兼容 API）做一次非流式补全
    response = ollama.chat.completions.create(model=MODEL, messages=messages)
    # 取出助手回复文本，交给 Gradio 显示
    return response.choices[0].message.content

# ChatInterface：把 chat 函数挂成对话界面；type="messages" 使用 OpenAI 风格历史
gr.ChatInterface(fn=chat, type="messages").launch()
